In [210]:
import cv2
import numpy as np

# Load the image
image = cv2.imread('test2jpg.jpg')

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Display the image (optional)
cv2.imshow('Grayscale Image', gray)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [211]:
# Estimate background with large kernel Gaussian blur
background = cv2.GaussianBlur(gray, (55, 55), 0)

# Subtract background to correct lighting
corrected = cv2.divide(gray, background, scale=255)

cv2.imshow("Light Normalized", corrected)
cv2.waitKey(0)


-1

In [212]:
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
enhanced = clahe.apply(corrected)

cv2.imshow("Contrast Enhanced", enhanced)
cv2.waitKey(0)


-1

In [213]:
# Denoise the enhanced image to remove grain
denoised = cv2.fastNlMeansDenoising(enhanced, h=25, templateWindowSize=7, searchWindowSize=21)

cv2.imshow("Final Denoised", denoised)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [214]:
# Initialize ORB detector
orb = cv2.ORB_create()

# Detect keypoints and descriptors
keypoints, descriptors = orb.detectAndCompute(denoised, None)

# Draw keypoints on the image
denoised_with_keypoints = cv2.drawKeypoints(denoised, keypoints, None)

cv2.imshow("ORB Keypoints", denoised_with_keypoints)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [215]:
import cv2
import numpy as np

# Resize to match model input (64x256)
resized = cv2.resize(denoised, (256, 64))  # (width, height)

# Normalize to [0, 1] and add batch and channel dims
input_tensor = resized.astype(np.float32) / 255.0
input_tensor = np.expand_dims(input_tensor, axis=0)    # Add batch dim
input_tensor = np.expand_dims(input_tensor, axis=0)    # Add channel dim

print("✅ Image preprocessed for ONNX input:", input_tensor.shape)


✅ Image preprocessed for ONNX input: (1, 1, 64, 256)


In [216]:
import onnxruntime as ort

# Load ONNX model
ort_session = ort.InferenceSession(r"Brahmi2\brahmi_ocr_model.onnx")

# Run inference
outputs = ort_session.run(None, {"input": input_tensor})[0]

print("📤 Raw model output shape:", outputs.shape)  # Expecting (seq_len, batch, num_classes)


📤 Raw model output shape: (32, 1, 111)


In [217]:
import pandas as pd

# Load metadata CSV
df = pd.read_csv(r"Brahmi2\dataset\metadata.csv")

# Extract all unique characters
all_text = ''.join(df['text'])
unique_chars = sorted(set(''.join(df['text'])))
print("len:", len(unique_chars))  # Should be 110
chars = unique_chars


len: 109


In [228]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from ctcdecode import CTCBeamDecoder

# Define the image path
image_path = r"Brahmi2\test_images\sampleBrahmi.jpg"

# Step 1: Preprocess the Image
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
if image is None:
    print("Error: Could not load image.")
    exit()

# Apply median blur to reduce noise
blurred = cv2.medianBlur(image, 3)

# Binarize using adaptive thresholding
thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

# Resize to a fixed height (e.g., 32 pixels) while preserving aspect ratio
target_height = 32
aspect_ratio = image.shape[1] / image.shape[0]
target_width = int(target_height * aspect_ratio)
resized = cv2.resize(thresh, (target_width, target_height), interpolation=cv2.INTER_AREA)

# Normalize to [0,1]
normalized = resized / 255.0

# Convert to PyTorch tensor (1, 1, H, W)
input_tensor = torch.from_numpy(normalized).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 32, W)

# Step 2: Load Your CRNN Model
# Replace with your actual model class and loading logic
# Example:
# class CRNNModel(torch.nn.Module):
#     def __init__(self, num_classes):
#         super(CRNNModel, self).__init__()
#         # Define your CNN, RNN, and linear layers
#         self.cnn = torch.nn.Conv2d(1, 64, kernel_size=3, padding=1)
#         self.rnn = torch.nn.LSTM(64, 128, bidirectional=True, num_layers=2)
#         self.linear = torch.nn.Linear(256, num_classes)
#
#     def forward(self, x):
#         # Implement forward pass
#         return x  # Placeholder
#
# model = CRNNModel(num_classes=111)
# model.load_state_dict(torch.load('epoch_10_checkpoint.pth'))
# model.eval()

# Step 3: Run Inference
# with torch.no_grad():
#     output = model(input_tensor)  # Shape: (T, 1, 111), e.g., (32, 1, 111)

# For now, use a placeholder output (replace with actual inference)
# Since we don't have the model, we'll simulate with zeros to demonstrate decoding
output = torch.zeros(32, 1, 111)  # Placeholder; replace with actual output

# Step 4: Apply Softmax to Get Probabilities
probs = F.softmax(output, dim=2)  # Shape: (32, 1, 111)

# Step 5: CTC Decoding with Beam Search
# Define the character set (excluding blank)
# Replace with your actual character mapping
brahmi_chars = [chr(0x11000 + i) for i in range(110)]  # U+11000 to U+1106D (placeholder)
decoder = CTCBeamDecoder(
    [''] + brahmi_chars,  # Include blank as first character
    beam_width=10,
    blank_id=0
)
beam_results, _, _, out_lens = decoder.decode(probs)
decoded_indices = beam_results[0][0][:out_lens[0][0]]

# Step 6: Map Indices to Brahmi Characters
# Replace with your actual mapping
brahmi_mapping = {i: chr(0x11000 + i - 1) for i in range(1, 111)}  # Placeholder mapping
brahmi_mapping[0] = ''  # Blank label

# Convert indices to Brahmi text
predicted_text = ''.join(brahmi_mapping.get(idx.item(), '') for idx in decoded_indices)

# Step 7: Output the Result
print("Predicted Brahmi Text:", predicted_text)

ModuleNotFoundError: No module named 'ctcdecode'

In [219]:
def segment_brahmi_characters(denoised_image):
    """Improved segmentation for Brahmi's connected characters"""
    # 1. Adaptive Thresholding
    binary = cv2.adaptiveThreshold(denoised_image, 255, 
                                 cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY_INV, 11, 2)
    
    # 2. Morphological Operations for Brahmi-specific cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    morphed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # 3. Find Contours with Brahmi-specific parameters
    contours, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 4. Character-like Contour Filtering
    char_boxes = []
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        aspect_ratio = w/h
        area = cv2.contourArea(cnt)
        
        # Brahmi-specific size filters (adjust based on your data)
        if (10 < w < 300 and 20 < h < 300 and 
            0.2 < aspect_ratio < 3.0 and 
            area > 100):
            char_boxes.append((x,y,w,h))
    
    # 5. Left-to-Right Sorting with vertical grouping
    char_boxes.sort(key=lambda b: (b[1]//20, b[0]))  # Group by approximate line
    
    return char_boxes

In [220]:
def extract_brahmi_features(image, char_boxes):
    """Brahmi-specific feature extraction"""
    features = []
    for (x,y,w,h) in char_boxes:
        char_img = image[y:y+h, x:x+w]
        
        # 1. Resize and normalize
        char_img = cv2.resize(char_img, (64,64))
        char_img = char_img.astype(np.float32)/255.0
        
        # 2. Structural Features
        moments = cv2.moments(char_img)
        hu_moments = cv2.HuMoments(moments).flatten()
        
        # 3. Projection Profiles
        horz_proj = cv2.reduce(char_img, 1, cv2.REDUCE_MEAN).flatten()
        vert_proj = cv2.reduce(char_img, 0, cv2.REDUCE_MEAN).flatten()
        
        # 4. Zone Densities (9-zone)
        zones = []
        for i in range(3):
            for j in range(3):
                zone = char_img[i*21:(i+1)*21, j*21:(j+1)*21]
                zones.append(np.mean(zone))
        
        features.append({
            'hu_moments': hu_moments,
            'projection': np.concatenate([horz_proj, vert_proj]),
            'zones': zones,
            'position': (x,y,w,h)
        })
    
    return features

In [221]:
def hybrid_brahmi_recognition(image_path, onnx_path, chars):
    # 1. Preprocessing
    denoised = preprocess_brahmi_image(image_path)  # Your existing function
    
    # 2. Character Segmentation
    char_boxes = segment_brahmi_characters(denoised)
    
    # 3. Feature Extraction
    features = extract_brahmi_features(denoised, char_boxes)
    
    # 4. Deep Learning Prediction
    ort_session = ort.InferenceSession(onnx_path)
    outputs = ort_session.run(None, {"input": denoised[np.newaxis, np.newaxis, ...]})[0]
    
    # 5. Feature-Enhanced Decoding
    predictions = []
    for t in range(min(outputs.shape[0], len(features))):
        char_probs = outputs[t,0,:]
        
        # Apply feature-based corrections
        if features[t]['zones'][4] > 0.7:  # Strong center zone
            char_probs[chars.index('𑀓')] *= 1.5  # Boost 'ka' probability
        
        # Use vertical projection to detect vowel signs
        if features[t]['projection'][64+32] > 0.8:  # Right-side projection
            char_probs[chars.index('𑀸')] *= 2.0  # Boost vowel sign
            
        predictions.append(np.argmax(char_probs))
    
    return ''.join([chars[i] for i in predictions if i < len(chars)])

In [222]:
def visualize_recognition(image_path, predictions, char_boxes):
    image = cv2.imread(image_path)
    for i, (x,y,w,h) in enumerate(char_boxes):
        if i < len(predictions):
            cv2.rectangle(image, (x,y), (x+w,y+h), (0,255,0), 2)
            cv2.putText(image, predictions[i], (x,y-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    
    cv2.imshow("Recognition Results", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [223]:
# Load your character set
chars = ['𑀓', '𑀔', '𑀕', '𑀖', '𑀗', '𑀸', '𑀹', '𑀺', ...]  # Complete your character list

# Process image
image_path = r"Brahmi2\test12.png"
prediction = hybrid_brahmi_recognition(image_path, r"Brahmi2\brahmi_ocr_model.onnx", chars)

# Get character boxes for visualization
denoised = preprocess_brahmi_image(image_path)
char_boxes = segment_brahmi_characters(denoised)

# Visualize results
visualize_recognition(image_path, list(prediction), char_boxes)
print("Final Prediction:", prediction)

error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\thresh.cpp:1679: error: (-215:Assertion failed) src.type() == CV_8UC1 in function 'cv::adaptiveThreshold'
